# 🫀 Medical LLM Assistant — Especialidade Cardiologia

Este colab executa o sistema no estado do commit `9807f07` (antes das refatorações de pipeline),
com o modelo de cardiologia treinado pelo Lucas.

**Diferenças do colab principal:**
- Foco em cardiologia (SCA, IC, arritmias)
- Pipeline simplificado (pré-refatoração)
- Modelo específico da especialidade

**Link do modelo:** https://drive.google.com/drive/folders/13_ASAfgA2r-mHXqNlW9xvIXj2FxNI85n

## 1️⃣ Clone do Repositório (Commit Específico)

In [7]:
import os, sys

# Garante diretório válido antes de operar
os.chdir('/content')

REPO_URL = "https://github.com/felipe-huszar/medical-llm-assistant.git"
REPO_DIR = "/content/medical-llm-assistant"
BRANCH = "cardio-specialist"

if os.path.exists(REPO_DIR):
    print("Repositório já existe — removendo...")
    import shutil
    shutil.rmtree(REPO_DIR)

print(f"Clonando repositório...")
!git clone -b {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

print(f"\n✅ Repositório pronto na branch {BRANCH}")
!git log --oneline -1


Repositório já existe — removendo...
Clonando repositório...
Cloning into '/content/medical-llm-assistant'...
remote: Enumerating objects: 713, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 713 (delta 102), reused 123 (delta 60), pack-reused 534 (from 1)
Receiving objects: 100% (713/713), 331.89 KiB | 1.85 MiB/s, done.
Resolving deltas: 100% (400/400), done.
/content/medical-llm-assistant
Note: switching to '9807f07'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHea

## 2️⃣ Instalação de Dependências

In [1]:
import importlib.util

if importlib.util.find_spec('chromadb') is None:
    print('📦 Instalando dependências...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'])
    subprocess.run(['pip', 'install', '-q', 'unsloth', 'peft', 'bitsandbytes', 'accelerate', 'gdown'])
    print('🔄 Reiniciando runtime... Execute esta célula novamente após o restart.')
    import os; os.kill(os.getpid(), 9)
else:
    print('✅ Dependências instaladas.')

✅ Dependências instaladas.


## 3️⃣ Configuração de Ambiente

In [ ]:
# Google Drive — montar para acessar modelo treinado
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado.')

In [2]:
import os, sys
from pathlib import Path

# ──────────────────────────────────────────────────────────
# Configuração do Modelo de Cardiologia (Lucas)
# ──────────────────────────────────────────────────────────
USE_MOCK = 'false'   # 'false' para usar modelo real

# Modelo de cardiologia do Lucas
# Link direto: https://drive.google.com/file/d/1bvoS95ulV5JgUAChf6up3DwIcjqdCUyx/view
MODEL_FILE_ID = '1bvoS95ulV5JgUAChf6up3DwIcjqdCUyx'
MODEL_ZIP_NAME = 'medical_cardiology_assistant_llm_14b.zip'
LORA_LOCAL_PATH = '/content/model'

os.environ['USE_MOCK_LLM'] = USE_MOCK
os.environ['MODEL_PATH'] = LORA_LOCAL_PATH
os.environ['BASE_MODEL_ID'] = 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit'
os.environ['CHROMA_DB_PATH'] = '/content/chroma_db'

sys.path.insert(0, '/content/medical-llm-assistant')

print('=' * 60)
print('  CONFIGURAÇÃO — ESPECIALIDADE CARDIOLOGIA')
print('=' * 60)
print(f"  USE_MOCK_LLM   = {os.environ['USE_MOCK_LLM']}")
print(f"  MODEL_PATH     = {os.environ['MODEL_PATH']}")
print(f"  BASE_MODEL_ID  = {os.environ['BASE_MODEL_ID']}")
print(f"  MODEL_FILE_ID  = {MODEL_FILE_ID}")
print('=' * 60)


  CONFIGURAÇÃO — ESPECIALIDADE CARDIOLOGIA
  USE_MOCK_LLM   = false
  MODEL_PATH     = /content/model
  BASE_MODEL_ID  = unsloth/Qwen2.5-14B-Instruct-bnb-4bit
  MODEL_FILE_ID  = 1bvoS95ulV5JgUAChf6up3DwIcjqdCUyx


In [3]:
if USE_MOCK.lower() != 'true':
    import subprocess
    import zipfile
    import shutil

    print('\n📥 Baixando modelo de cardiologia do Drive do Lucas...')
    os.makedirs(LORA_LOCAL_PATH, exist_ok=True)
    zip_path = f'{LORA_LOCAL_PATH}/{MODEL_ZIP_NAME}'

    # Download direto via gdown usando file ID
    print(f'⬇️  Download: {MODEL_ZIP_NAME}')
    result = subprocess.run(
        ['gdown', f'https://drive.google.com/uc?id={MODEL_FILE_ID}', '-O', zip_path],
        capture_output=True, text=True
    )

    if result.returncode != 0 or not os.path.exists(zip_path):
        print('❌ Erro no download:', result.stderr)
        print('\n💡 Tentando buscar no Drive montado...')

        # Fallback: busca no Drive montado
        search_paths = [
            '/content/drive/MyDrive',
            '/content/drive/MyDrive/FIAP-TC-Fase3',
            '/content/drive/MyDrive/FIAP-TC-Fase3/models',
        ]
        found = False
        for search_path in search_paths:
            candidate = f'{search_path}/{MODEL_ZIP_NAME}'
            if os.path.exists(candidate):
                print(f'✅ Encontrado no Drive: {candidate}')
                shutil.copy(candidate, zip_path)
                found = True
                break

        if not found:
            raise FileNotFoundError(f'Modelo não encontrado. Verifique o link do Drive.')
    else:
        print('✅ Download concluído.')

    print(f'\n📦 Extraindo: {MODEL_ZIP_NAME}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(LORA_LOCAL_PATH)
    print('✅ Modelo extraido.')

    # Valida estrutura do adapter
    cfg = None
    wts = None
    for root, _, files in os.walk(LORA_LOCAL_PATH):
        if 'adapter_config.json' in files and 'adapter_model.safetensors' in files:
            cfg = os.path.join(root, 'adapter_config.json')
            wts = os.path.join(root, 'adapter_model.safetensors')
            break

    if cfg and wts:
        final_model_path = os.path.dirname(cfg)
        os.environ['MODEL_PATH'] = final_model_path
        print('\n✅ Modelo validado com adapter LoRA.')
        print(f'   MODEL_PATH = {final_model_path}')
    else:
        print('❌ Erro: adapter_config.json ou adapter_model.safetensors não encontrados')
        print(f'\nConteúdo de {LORA_LOCAL_PATH}:')
        !find {LORA_LOCAL_PATH} -maxdepth 3 -type f | sort
        raise RuntimeError('Artefato inválido para o loader (esperado: adapter-only).')
else:
    print('\n🎭 Modo Mock ativo — modelo não necessário.')



📥 Baixando modelo de cardiologia do Drive do Lucas...
⬇️  Download: medical_cardiology_assistant_llm_14b.zip
✅ Download concluído.

📦 Extraindo: medical_cardiology_assistant_llm_14b.zip
✅ Modelo extraido.

✅ Modelo validado com adapter LoRA.
   MODEL_PATH = /content/model


## ⚠️ Limpeza do ChromaDB (Opcional)

In [4]:
# ⚠️ LIMPEZA TOTAL DO CHROMADB — execute apenas se tiver certeza!
CONFIRMAR = False  # mude para True para confirmar

if CONFIRMAR:
    import src.rag.patient_rag as rag
    client = rag._get_client()
    for col_name in ['patients', 'consultations']:
        try:
            client.delete_collection(col_name)
            print(f'✅ Collection {col_name!r} removida')
        except Exception:
            print(f'ℹ️  Collection {col_name!r} não existia')
    rag._client = None
    rag._client_path = None
    from src.rag.patient_rag import seed_from_file
    n = seed_from_file('data/patients_seed.json')
    print(f'✅ {n} pacientes seed reinseridos. ChromaDB limpo!')
else:
    print('⚠️  Limpeza NÃO executada. Mude CONFIRMAR = True para prosseguir.')

⚠️  Limpeza NÃO executada. Mude CONFIRMAR = True para prosseguir.


## 4️⃣ Carregar Modelo na Memória

In [8]:
import os, src.llm.factory as _factory

if os.environ.get('USE_MOCK_LLM', 'true').lower() != 'true':
    if _factory._cached_llm is None:
        print('🔄 Carregando modelo de cardiologia na memória...')
        _factory.get_llm()
        print('✅ Modelo pronto e em cache.')
    else:
        print('✅ Modelo já estava em cache.')
else:
    print('🎭 Modo Mock ativo.')

✅ Modelo já estava em cache.


## 5️⃣ Interface Gradio — Cardiologia

In [9]:
import sys

# Limpa cache exceto src.llm.* (modelo em VRAM)
for mod in list(sys.modules):
    if mod == "app" or (mod.startswith("src.") and not mod.startswith("src.llm")):
        del sys.modules[mod]

import app as _app
import src.llm.factory as _factory

if _factory._cached_llm is not None:
    _app.set_llm(_factory._cached_llm)
    print('✅ Modelo de cardiologia conectado à UI.')
else:
    print('⚠️ Modelo não está em cache. Rode a célula anterior primeiro.')

print('\n🚀 Iniciando interface...')
_app.demo.queue().launch(share=True)

✅ Modelo de cardiologia conectado à UI.

🚀 Iniciando interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://88a6ff954c06da8fd9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

## 📝 Notas de Uso

**Este colab usa:**
- Commit `9807f07` do repositório (estado pré-refatoração)
- Modelo de cardiologia treinado pelo Lucas
- Pipeline simplificado sem benchmark mode

**Para comparação com o sistema principal:**
- Use `system_gradio.ipynb` para o modelo geral (pipeline refinado)
- Use este colab para o modelo de cardiologia (pipeline original)